In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
import time
import os
import scipy.signal # power spectral density fxn.
import datetime
import socket
import glob

In [ ]:
#file_list = glob.glob('IN_h5/1chfw_1000tones_direct*seconds.hd5') #test file
# file_list = glob.glob('./*.hd5') #all hd5 file in input directory
import socket
addr = ("192.168.3.40", 4096)
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind(addr)
sock.settimeout(3)

def parse_packet():
    data = sock.recv(8192)
    if len(data) <  8000:
        print("invalid packet recieved")
        return
    datarray = bytearray(data)

    # now allow a shift of the bytes
    spec_data = np.frombuffer(datarray, dtype = '<i', offset=0x00)
    # offset allows a shift in the bytes
    return spec_data # int32 data type

def capture_packets(N_packets):
    # toss out buffer data
    for i in range(2000):
        parse_packet()
    packets = np.zeros(shape=(2048,N_packets))
    #packets = np.zeros(shape=(2051,N_packets))
    counter = 0
    for i in range(N_packets):
        data_2 = parse_packet()
        packets[:,i] = data_2 
        if i%488 == 0:
            print("{}/{} captured ({:.3f}% Complete)".format(i, N_packets, 
                (i/N_packets)*100.0))
    print("{}/{} captured ({:.3f}% Complete)".format(N_packets, N_packets, (N_packets/N_packets)*100.0))
    return packets
    

In [ ]:
data = None
try:
    data = capture_packets(488*20)
except TimeoutError:
    raise TimeoutError("Are we receiving data and are the MAC addresses correct?")
    sock.close()
I=data[::2]
Q=data[1::2]
filename = "/data/NOISEEEEE/2chanfw_......"
np.save(filename, np.array((I,Q)))

In [ ]:
# # filename = "/data/NOISEEEEE/SanityCheckDualChannel_noamp-noeq-chan1-1000tones"
# #filename = "exclaimifloopback-10tones-1ghzlo-29-dbatten"'
# # I,Q = np.load(filename + '.npy')
# plt.figure()
# MYI = I[8][0:100].astype("double")
# MYQ = Q[8][0:100].astype("double")
# # plt.plot(MYI)
# # plt.plot(MYQ)
# # plt.figure()
# plt.plot((MYI**2+MYQ**2)/1e14)



In [ ]:
def template(I,Q):
    # subtract the mean from each detector
    Imeansub = np.zeros_like(I)
    Qmeansub = np.zeros_like(Q)
    for i in range(len(I[:,0])):
        Imeansub[i,:] = I[i,:] - np.mean(I[i,:])
        Qmeansub[i,:] = Q[i,:] - np.mean(Q[i,:])
    
    # select only the middle few detectors
    deproj_I = Imeansub[8:1008 ,:]
    deproj_Q = Qmeansub[8:1008,:]

    # create a separate correlation matrix for I and Q
    correlation_matrix_I = np.matmul(deproj_I,np.conj(np.transpose(deproj_I)))
    correlation_matrix_Q = np.matmul(deproj_Q,np.conj(np.transpose(deproj_Q)))
    # calculate the eigenmodes of each correlation matrix
    wI,vI = np.linalg.eig(correlation_matrix_I)
    wQ,vQ = np.linalg.eig(correlation_matrix_Q)
    # create templates based on the largest eigenmode of each
    templateI0 = np.matmul(vI[:,0],deproj_I)    
    templateQ0 = np.matmul(vQ[:,0],deproj_Q)

    # subtract the mean again to be sure
    template_real0 = np.real(templateI0)-np.mean(np.real(templateI0))
    template_imag0 = np.real(templateQ0)-np.mean(np.real(templateQ0))

    # create templates based on the second largest eigenmode of each
    templateI1 = np.matmul(vI[:,1],deproj_I)    
    templateQ1 = np.matmul(vQ[:,1],deproj_Q)

    # subtract the mean again to be sure
    template_real1 = np.real(templateI1)-np.mean(np.real(templateI1))
    template_imag1 = np.real(templateQ1)-np.mean(np.real(templateQ1))
   
    #plt.figure()
    #plt.subplot(2,1,1)
    #plt.semilogy(wI,"x")
    #plt.grid("on")
    #plt.subplot(2,1,2)
    #plt.semilogy(wQ,"x")
    #plt.grid("on")
    
    return template_real0,template_imag0,template_real1,template_imag1


In [ ]:
def clean(I,Q,template_I0,template_Q0,template_I1,template_Q1):
    Iclean = np.zeros_like(I)
    Qclean = np.zeros_like(Q)
    
    for idet in range(len(I[:,0])):
        samp_chan_I = I[idet,:]
        samp_chan_Q = Q[idet,:]
    
        corr0 = np.matmul(samp_chan_I,np.transpose(template_I0))/np.matmul(template_I0,np.transpose(template_I0))
        deprojected_samp_detector_I = samp_chan_I-corr0*template_I0
        corr1 = np.matmul(deprojected_samp_detector_I,np.transpose(template_I1))/np.matmul(template_I1,np.transpose(template_I1))
        deprojected_samp_detector_I = deprojected_samp_detector_I-corr1*template_I1
        
        corr0 = np.matmul(samp_chan_Q,np.transpose(template_Q0))/np.matmul(template_Q0,np.transpose(template_Q0))
        deprojected_samp_detector_Q = samp_chan_Q-corr0*template_Q0
        corr1 = np.matmul(deprojected_samp_detector_Q,np.transpose(template_Q1))/np.matmul(template_Q1,np.transpose(template_Q1))
        deprojected_samp_detector_Q = deprojected_samp_detector_Q-corr1*template_Q1
    
        Iclean[idet,:] = deprojected_samp_detector_I
        Qclean[idet,:] = deprojected_samp_detector_Q
    return Iclean,Qclean


In [ ]:
def noise_plot(I,Q,npoints,file):
    # initialize variables
    nfs = int(np.log2(npoints))
    f_interp = np.zeros(nfs)
    Spp_i_interp = np.zeros((1024,nfs))
    Spp_q_interp = np.zeros((1024,nfs))
    Spp_interp = np.zeros((1024,nfs))

    # find the power in each detector
    rms_per_det = np.std(I,axis=1) + np.std(Q,axis=1)
    # find the detector with median power
    idet_median = np.argsort(rms_per_det)[int(len(rms_per_det)/2)]
    plt.figure(figsize=(12,8))
    #for idet in range(4,14):
    for idet in range(len(I[:,0])):
        # form the complex signal of this detector
        Z = I[idet,:] + 1j*Q[idet,:]

        # calculate the DC value for this detector
        norm = np.mean(np.abs(Z))

        # take the Welch periodogram (viewer-friendly FFT)
        # of the real and imaginary parts of the signal
        f,Spp_i=scipy.signal.welch(np.real(Z)/norm,fs=512e6/2**(10+10),nperseg=npoints)
        f,Spp_q=scipy.signal.welch(np.imag(Z)/norm,fs=512e6/2**(10+10),nperseg=npoints)

        # smooth the curve further for plotting
        for j in range(0,nfs-1):
            f_interp[j] = np.mean(f[2**j:2**(j+1)])
            Spp_i_interp[idet,j] = np.mean(Spp_i[2**j:2**(j+1)])
            Spp_q_interp[idet,j] = np.mean(Spp_q[2**j:2**(j+1)])

        # combine I and Q spectra for plotting
        Spp_interp[idet,:] = (Spp_i_interp[idet,:] + Spp_q_interp[idet,:])/2

        # plot this detector
        plt.semilogx(f_interp[:],10*np.log10(Spp_i_interp[idet,:])[:],"-",color="cyan",alpha=0.2)#,label="All Detectors")
        plt.semilogx(f_interp[:],10*np.log10(Spp_q_interp[idet,:])[:],"-",color="cyan",alpha=0.2)#,label="All Detectors")
        # plot non smoothed data
        # plt.semilogx(f[:],10*np.log10(Spp_i[:])[:],"-",color="blue",alpha=0.04)#,label="All Detectors")
        # plt.semilogx(f[:],10*np.log10(Spp_q[:])[:],"-",color="orange",alpha=0.04)#,label="All Detectors")
        # if we are at the detector with median power
        if idet == idet_median:
            # plot this again, but with a solid black line to highlight it
            plt.semilogx(f_interp[:],10*np.log10(Spp_interp[idet,:])[:],"-",color="black",alpha=1.00,label="Median Detectors", zorder=99999)
    
    # fname = 'OUT/'+file.split('/')[1].split('.')[0]
    fname = file+".png"
    #fname = filename
    
    plt.title(file)# 20-900 Channel 2")
    plt.ylabel(r"$S_{\phi \phi}$ [dBc/Hz]", fontsize=16); 
    plt.xlabel("Hz", fontsize=16)
    plt.ylim(-140,-40)
    plt.yticks(np.linspace(-40, -140, 21))
    #plt.yticks(np.linspace(-110, -50, abs(110-50)-1))
    plt.grid()
    plt.savefig(fname)
    plt.show()
    print(fname)
    return

In [ ]:
# Grab all of the measured data, process, and plot it
datasets = glob.glob('*.npy')
[print(f"{a}.", b) for a,b in enumerate(datasets)]
print("")

for ix in datasets:
    f = ix.replace('.npy', '')
    I,Q = np.load(ix)
    template_I0,template_Q0,template_I1,template_Q1 = template(I,Q) #create templates from first two modes
    Iclean,Qclean = clean(I,Q,template_I0,template_Q0,template_I1,template_Q1) #subtract both modes from I and Q
    npoints = 8192
    noise_plot(I[8:1008],Q[8:1008],npoints,f) #make d

In [ ]:
sock.close()